# Combined original feature, LGB, NN and rounding corrections; fixed seed retained

## 可核查的实现选择

- 本版将补充集第 5 月改为第 6 月，训练起点按冠军源码限定为有临床记录的指定月份访视。匹配 0、6、12、24 月后有效标签；未发生的目标访视不补造标签。目标月份不额外截断。患者 ID 只用于连接和划分，不输入模型。
- 第 6、18、48 月的访视状态：预测起点早于该月时为 -1（尚不可知），到达该月后为 0/1（未见/已见）。非年度访视数只统计起点**以前**的第 6、18 月；当前访视由对应状态特征表示。未来记录不会改变过去的特征。以上编码是本次选择，原文未明确指定。
- 补充集没有采血元数据，采血标记设 0，另用补充集指示器区别来源；不填补组学测量。
- LightGBM 固定 87 类（0–86），在完整 87 类概率分布上，对 87 个候选整数计算期望损失并取最小者；不能用概率均值或先 argmax 代替。
- MLP：两层 128 隐藏单元；隐藏层和最终输出均为 LeakyReLU。连续特征按训练折标准化，类别使用预定义 one-hot；训练 160 个 epoch，Adam + cosine scheduler，学习率 0.003。这些是独立选定参数，未声称来自冠军源码。NN 最终输出裁剪至 0–86；融合后保留小数。
- SMAPE+1 = `mean(200 * abs(y-p) / (2 + abs(y) + abs(p)))`；两边各加 1，所以分母含 2。
- 原文最终验证为留一患者。这里默认先执行 5 折患者 GroupKFold；设 `CONFIG['cv'] = 'lopo'` 即对 248 位主集患者逐一留出。补充集始终仅参与训练。评分只包含主集实际采血起点的请求，同时报告五折均值和合并 OOF。**5 折结果不是留一患者结果，也不是 Kaggle 分数。** 未做超参数搜索。


In [ ]:
"""Independent implementation of the Connecting dotts writeup; no solution source used."""


## Imports and configuration

In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
import json
import random
import time
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
from torch import nn
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler

TARGETS = ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']
HORIZONS = (0, 6, 12, 24)
FEATURES = ['visit_month', 'horizon', 'target_month', 'blood_taken',
            'supplemental', 'seen_6', 'seen_18', 'seen_48',
            'previous_nonannual', 'target_index']
CONTINUOUS = ['visit_month', 'horizon', 'target_month', 'previous_nonannual']
CATEGORIES = {'blood_taken': [0, 1], 'supplemental': [0, 1],
              'seen_6': [-1, 0, 1], 'seen_18': [-1, 0, 1],
              'seen_48': [-1, 0, 1], 'target_index': [0, 1, 2, 3],
              'horizon': [0, 6, 12, 24]}
CONFIG = dict(seed=4486, folds=5, cv='group', lgb_rounds=300,
              epochs=160, hidden=128, learning_rate=0.003, batch_size=512)
torch.set_num_threads(1)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def smape1p(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(np.mean(200 * np.abs(y - p) / (2 + np.abs(y) + np.abs(p))))



## Causal features and supervised request table

In [ ]:
def load_data(data_dir):
    data_dir = Path(data_dir)
    clinical = pd.read_csv(data_dir / 'train_clinical_data.csv')
    supplemental = pd.read_csv(data_dir / 'supplemental_clinical_data.csv')
    supplemental.loc[supplemental['visit_month'] == 5, 'visit_month'] = 6
    # Only visit identifiers are read, never protein/peptide concentrations.
    blood = pd.read_csv(data_dir / 'train_proteins.csv',
                        usecols=['patient_id', 'visit_month']).drop_duplicates()
    clinical['supplemental'] = 0
    supplemental['supplemental'] = 1
    assert set(clinical.patient_id).isdisjoint(set(supplemental.patient_id))
    visits = pd.concat([clinical, supplemental], ignore_index=True)
    blood['blood_taken'] = 1
    # Retain the V24 visit history; blood-only rows are excluded as training origins below.
    visits = visits.merge(blood, on=['patient_id', 'visit_month'], how='outer',
                          validate='one_to_one')
    visits['supplemental'] = visits.supplemental.fillna(0).astype(int)
    visits['blood_taken'] = visits.blood_taken.fillna(0).astype(int)
    assert not visits.duplicated(['patient_id', 'visit_month']).any()
    return visits

def causal_features(requests, history):
    """History may contain future rows: every lookup is explicitly time-censored."""
    out = requests.copy()
    if 'visit_id' in history.columns:
        history = history.loc[history.visit_id.notna()]
    timelines = {int(pid): set(g.visit_month.astype(int))
                 for pid, g in history.groupby('patient_id')}
    for marker in (6, 18, 48):
        # -1: marker is still in the future; 0: not observed; 1: observed.
        out['seen_' + str(marker)] = [
            int(month >= marker and marker in timelines.get(int(pid), set()))
            for pid, month in zip(out.patient_id, out.visit_month)]
    out['previous_nonannual'] = [
        sum(m <= month and m in timelines.get(int(pid), set()) for m in (6, 18))
        for pid, month in zip(out.patient_id, out.visit_month)]
    out['target_month'] = out.visit_month + out.horizon
    return out

def build_samples(visits):
    origin_months = (0, 6, 12, 18, 24, 36, 48, 60, 72, 84)
    eligible = visits.visit_id.notna() & visits.visit_month.isin(origin_months)
    anchors = visits.loc[eligible, ['patient_id', 'visit_month', 'supplemental', 'blood_taken']]
    anchors = pd.concat([anchors.assign(horizon=h) for h in HORIZONS], ignore_index=True)
    anchors['target_month'] = anchors.visit_month + anchors.horizon
    labels = visits[['patient_id', 'visit_month'] + TARGETS].rename(
        columns={'visit_month': 'target_month'})
    joined = anchors.merge(labels, on=['patient_id', 'target_month'], how='inner',
                           validate='many_to_one')
    samples = joined.melt(id_vars=list(anchors.columns), value_vars=TARGETS,
                          var_name='target', value_name='label').dropna(subset=['label'])
    samples['target_index'] = samples.target.map(dict(zip(TARGETS, range(4))))
    samples = causal_features(samples, visits).reset_index(drop=True)
    assert samples.label.between(0, 86).all()
    assert np.equal(samples.label, samples.label.astype(int)).all()
    samples['score_eligible'] = (samples.supplemental == 0) & (samples.blood_taken == 1)
    return samples



## Discrete Bayes action for SMAPE+1

In [ ]:
VALUES = np.arange(87, dtype=np.float64)
# Rows are true classes, columns are candidate predictions.
COST = 200 * np.abs(VALUES[:, None] - VALUES[None, :]) / (
    2 + VALUES[:, None] + VALUES[None, :])

def optimal_integer(probabilities):
    probabilities = np.asarray(probabilities).reshape(-1, 87)
    assert np.isfinite(probabilities).all()
    return (probabilities @ COST).argmin(axis=1).astype(float)

def fit_lgb(frame, cfg):
    params = dict(objective='multiclass', num_class=87, metric='multi_logloss',
                  learning_rate=0.019673004699536346, num_leaves=208, min_data_in_leaf=850,
                  lambda_l1=7.405660751699475e-08, lambda_l2=0.14583961675675494,
                  max_depth=14, max_bin=240, feature_fraction=0.5190632906197453, verbosity=-1,
                  num_threads=2, seed=cfg['seed'], deterministic=True,
                  force_col_wise=True)
    return lgb.train(params, lgb.Dataset(frame[FEATURES], label=frame.label.astype(int)),
                     num_boost_round=cfg['lgb_rounds'])



## Neural network: fold-local scaling and direct competition loss

In [ ]:
import numpy as np
import pandas as pd
import sys
import random
from tqdm import tqdm
import gc
import torch
from torch import optim
from torch.cuda.amp import GradScaler, autocast
from collections import defaultdict
from copy import copy
import os
import math
def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def multiplier(step):
        if step < num_warmup_steps:
            return float(step) / max(1, num_warmup_steps)
        progress = float(step-num_warmup_steps) / max(1,num_training_steps-num_warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)
from torch.utils.data import SequentialSampler, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, cohen_kappa_score
from torch.utils.data import Dataset
from scipy.special import softmax

torch.set_num_threads(1)

def single_smape1p(preds, tgt):
    x = np.tile(np.arange(preds.shape[1]), (preds.shape[0], 1))
    x = np.abs(x - tgt) / (2 + x + tgt)
    return (x * preds).sum(axis=1)

def opt_smape1p(preds):
    x = np.hstack([single_smape1p(preds, i).reshape(-1,1) for i in range(preds.shape[1])])
    return x.argmin(axis=1)


from types import SimpleNamespace
from torch import nn
import torch

class CustomDataset(Dataset):
    def __init__(self, df, cfg, aug, mode="train"):
        self.cfg = cfg
        self.mode = mode
        self.df = df.copy()
        self.features = df[cfg.features].values
        if self.mode != "test":
            self.targets = df[self.cfg.target_column].values.astype(np.float32)
        else:
            self.targets = np.zeros(len(df))

    def __getitem__(self, idx):
        features = self.features[idx]
        targets = self.targets[idx]
        
        feature_dict = {
            "input": torch.tensor(features),
            "target_norm": torch.tensor(targets),
        }
        return feature_dict

    def __len__(self):
        return len(self.df)


class Net(nn.Module):
    def __init__(self, cfg):
        super(Net, self).__init__()
        self.cfg = cfg
        self.n_classes = cfg.n_classes
        self.cnn = nn.Sequential(*([
            nn.Linear(len(self.cfg.features), cfg.n_hidden),
            nn.LeakyReLU(),
            ] +
            [
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.LeakyReLU(),
            ] * self.cfg.n_layers)
        )

        self.head = nn.Sequential(
            nn.Linear(cfg.n_hidden, self.n_classes),
            nn.LeakyReLU(),
        )

    def forward(self, batch):
        input = batch["input"].float()
        y = batch["target_norm"]
        x = input
        x = self.cnn(x)
        preds = self.head(x).squeeze(-1)
        loss = (torch.abs(y - preds) / (torch.abs(0.01 + y) + torch.abs(0.01 + preds))).mean()
        return {"loss": loss, "preds": preds, "target_norm": y}


def worker_init_fn(worker_id):
    np.random.seed(np.random.get_state()[1][0] + worker_id)

def get_train_dataloader(train_ds, cfg, verbose):
    train_dataloader = DataLoader(
        train_ds,
        sampler=None,
        shuffle=True,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        pin_memory=False,
        collate_fn=cfg.tr_collate_fn,
        drop_last=cfg.drop_last,
        worker_init_fn=worker_init_fn,
    )
    if verbose:
        print(f"train: dataset {len(train_ds)}, dataloader {len(train_dataloader)}")
    return train_dataloader


def get_val_dataloader(val_ds, cfg, verbose):
    sampler = SequentialSampler(val_ds)
    if cfg.batch_size_val is not None:
        batch_size = cfg.batch_size_val
    else:
        batch_size = cfg.batch_size
    val_dataloader = DataLoader(
        val_ds,
        sampler=sampler,
        batch_size=batch_size,
        num_workers=cfg.num_workers,
        pin_memory=False,
        collate_fn=cfg.val_collate_fn,
        worker_init_fn=worker_init_fn,
    )
    if verbose:
        print(f"valid: dataset {len(val_ds)}, dataloader {len(val_dataloader)}")
    return val_dataloader


def get_scheduler(cfg, optimizer, total_steps):
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=cfg.warmup * (total_steps // cfg.batch_size),
        num_training_steps=cfg.epochs * (total_steps // cfg.batch_size),
    )
    return scheduler


def set_seed(seed=1234):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    #torch.backends.cudnn.deterministic = False
    #torch.backends.cudnn.benchmark = True

def batch_to_device(batch, device):
    batch_dict = {key: batch[key].to(device) for key in batch}
    return batch_dict

def run_eval(model, val_dataloader, cfg, pre="val", verbose=True):
    model.eval()
    torch.set_grad_enabled(False)
    val_data = defaultdict(list)
    if verbose:
        progress_bar = tqdm(val_dataloader)
    else:
        progress_bar = val_dataloader
    for data in progress_bar:
        batch = batch_to_device(data, cfg.device)
        if cfg.mixed_precision:
            with autocast():
                output = model(batch)
        else:
            output = model(batch)
        for key, val in output.items():
            val_data[key] += [output[key]]
    for key, val in output.items():
        value = val_data[key]
        if len(value[0].shape) == 0:
            val_data[key] = torch.stack(value)
        else:
            val_data[key] = torch.cat(value, dim=0)

    preds = val_data["preds"].cpu().numpy()
    if (pre == "val") and verbose:
        metric = smape1p(100*val_data["target_norm"].cpu().numpy(), 100*preds)
        print(f"{pre}_metric 1 ", metric)
        metric = smape1p(100*val_data["target_norm"].cpu().numpy(), np.round(100*preds))
        print(f"{pre}_metric 2 ", metric)
    
    return 100*preds


def run_train(cfg, train_df, val_df, test_df=None, verbose=True):
    os.makedirs(str(cfg.output_dir + "/"), exist_ok=True)

    if cfg.seed < 0:
        cfg.seed = np.random.randint(1_000_000)
    if verbose:
        print("seed", cfg.seed)
    set_seed(cfg.seed)

    train_dataset = CustomDataset(train_df, cfg, aug=None, mode="train")
    train_dataloader = get_train_dataloader(train_dataset, cfg, verbose)
    
    if val_df is not None:
        val_dataset = CustomDataset(val_df, cfg, aug=None, mode="val")
        val_dataloader = get_val_dataloader(val_dataset, cfg, verbose)

    if test_df is not None:
        test_dataset = CustomDataset(test_df, cfg, aug=None, mode="test")
        test_dataloader = get_val_dataloader(test_dataset, cfg, verbose)

    model = Net(cfg)
    model.to(cfg.device)

    total_steps = len(train_dataset)
    params = model.parameters()
    optimizer = optim.Adam(params, lr=cfg.lr, weight_decay=0)
    scheduler = get_scheduler(cfg, optimizer, total_steps)

    if cfg.mixed_precision:
        scaler = GradScaler()
    else:
        scaler = None

    cfg.curr_step = 0
    i = 0
    optimizer.zero_grad()
    for epoch in range(cfg.epochs):
        set_seed(cfg.seed + epoch)
        if verbose:
            print("EPOCH:", epoch)
            progress_bar = tqdm(range(len(train_dataloader)))
        else:
            progress_bar = range(len(train_dataloader))
        tr_it = iter(train_dataloader)
        losses = []
        gc.collect()

        for itr in progress_bar:
            i += 1
            data = next(tr_it)
            model.train()
            torch.set_grad_enabled(True)
            batch = batch_to_device(data, cfg.device)
            if cfg.mixed_precision:
                with autocast():
                    output_dict = model(batch)
            else:
                output_dict = model(batch)
            loss = output_dict["loss"]
            losses.append(loss.item())
            if cfg.mixed_precision:
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
                optimizer.step()
                optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()
        if val_df is not None:
            if (epoch + 1) % cfg.eval_epochs == 0 or (epoch + 1) == cfg.epochs:
                run_eval(model, val_dataloader, cfg, pre="val", verbose=verbose)

    if test_df is not None:
        return run_eval(model, test_dataloader, cfg, pre="test", verbose=verbose)
    else:
        return model

def run_test(model, cfg, test_df):
    test_dataset = CustomDataset(test_df, cfg, aug=None, mode="test")
    test_dataloader = get_val_dataloader(test_dataset, cfg, verbose=False)
    return run_eval(model, test_dataloader, cfg, pre="test", verbose=False)


class Encoder:
    def fit(self, frame):
        return self
    def transform(self, frame):
        months = (0,6,12,18,24,36,48,60,72,84)
        blocks = [frame[['seen_6']].to_numpy()]
        blocks += [(frame.target_month.to_numpy()[:,None] == np.array(months)).astype(float)]
        blocks += [(frame.visit_month.to_numpy()[:,None] == np.array(months)).astype(float)]
        blocks += [(frame.horizon.to_numpy()[:,None] == np.array(HORIZONS)).astype(float)]
        blocks += [(frame.target_index.to_numpy()[:,None] == np.arange(4)).astype(float)]
        blocks += [frame[['seen_18','seen_48','supplemental']].to_numpy()]
        blocks += [frame[['horizon']].to_numpy()/24.0]
        return np.concatenate(blocks,axis=1).astype(np.float32)
class PredictionNetwork(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net=net
    def forward(self,x):
        return 100*self.net.head(self.net.cnn(x)).squeeze(-1)
def fit_nn(frame,cfg):
    encoder=Encoder().fit(frame)
    x=encoder.transform(frame)
    cols=['f'+str(i) for i in range(x.shape[1])]
    train=pd.DataFrame(x,columns=cols)
    train['target_norm']=frame.label.to_numpy()/100.0
    settings=SimpleNamespace(target_column='target_norm',output_dir='nn_temp',seed=cfg['seed'],
        eval_epochs=1,mixed_precision=False,device='cpu',n_classes=1,batch_size=128,
        batch_size_val=256,n_hidden=64,n_layers=2,num_workers=0,drop_last=False,
        gradient_clip=1.0,lr=0.002,warmup=0,epochs=10,features=cols,
        tr_collate_fn=None,val_collate_fn=None)
    net=run_train(settings,train,None,None,verbose=False)
    model=PredictionNetwork(net)
    model.eval()
    return model,encoder

def predict_models(bundle, frame):
    tree, network, encoder = bundle
    a = optimal_integer(tree.predict(frame[FEATURES]))
    with torch.no_grad():
        b = network(torch.from_numpy(encoder.transform(frame))).numpy().clip(0, 86)
    return a, np.round(b)

def fit_models(frame, cfg):
    return (fit_lgb(frame, cfg),) + fit_nn(frame, cfg)



## Patient-level validation (supplemental patients never enter the score)

In [ ]:
def cross_validate(samples, cfg):
    # Split the 248 main patients. Supplemental data are training-only in every fold.
    main = samples.loc[samples.supplemental == 0]
    supplement = samples.loc[samples.supplemental == 1]
    splitter = LeaveOneGroupOut() if cfg['cv'] == 'lopo' else GroupKFold(cfg['folds'])
    oof = pd.DataFrame(index=main.index, columns=['lgb', 'nn', 'blend'], dtype=float)
    records = []
    for fold, (it, iv) in enumerate(splitter.split(main, groups=main.patient_id)):
        started = time.time()
        train = pd.concat([main.iloc[it], supplement], ignore_index=True)
        valid = main.iloc[iv]
        assert set(train.patient_id).isdisjoint(set(valid.patient_id))
        bundle = fit_models(train, cfg)
        a, b = predict_models(bundle, valid)
        oof.loc[valid.index, :] = np.column_stack([a, b, np.round((a + b) / 2)])
        mask = valid.score_eligible.to_numpy()
        record = {'fold': fold, 'patients': int(valid.patient_id.nunique()),
                  'scored_rows': int(mask.sum()), 'seconds': round(time.time() - started, 2)}
        for name in oof:
            record[name] = smape1p(valid.label.to_numpy()[mask],
                                  oof.loc[valid.index, name].to_numpy()[mask])
        records.append(record)
        print(json.dumps(record), flush=True)
    assert oof.notna().all().all()
    report = {'config': cfg, 'main_patients': int(main.patient_id.nunique()),
              'supplemental_patients': int(supplement.patient_id.nunique()),
              'training_rows': len(samples), 'scored_rows': int(main.score_eligible.sum()),
              'folds': records, 'pooled_oof': {}, 'mean_fold': {}}
    mask = main.score_eligible
    for name in oof:
        report['pooled_oof'][name] = smape1p(main.loc[mask, 'label'], oof.loc[main.index[mask], name])
        report['mean_fold'][name] = float(np.mean([r[name] for r in records]))
    return report, main.join(oof)



## Stateful inference, shared by local replay and the official API

In [ ]:
class Predictor:
    def __init__(self, bundle):
        self.bundle = bundle
        self.history = pd.DataFrame(columns=['patient_id', 'visit_month'])

    def predict_batch(self, test, peptides, proteins, sample):
        visits = test[['patient_id', 'visit_month']].drop_duplicates()
        self.history = pd.concat([self.history, visits], ignore_index=True).drop_duplicates()
        ids = sample.prediction_id.str.extract(
            r'^(\d+)_(\d+)_updrs_([1-4])_plus_(0|6|12|24)_months$')
        if ids.isna().any().any():
            raise ValueError('Unrecognized prediction_id')
        requests = pd.DataFrame({'patient_id': ids[0].astype(int),
                                 'visit_month': ids[1].astype(int),
                                 'target_index': ids[2].astype(int) - 1,
                                 'horizon': ids[3].astype(int), 'supplemental': 0})
        assert set(zip(requests.patient_id, requests.visit_month)).issubset(
            set(zip(visits.patient_id, visits.visit_month)))
        blood = set()
        for table in (proteins, peptides):
            if len(table):
                blood.update(zip(table.patient_id.astype(int), table.visit_month.astype(int)))
        requests['blood_taken'] = [int((p, m) in blood)
                                    for p, m in zip(requests.patient_id, requests.visit_month)]
        requests = causal_features(requests, self.history)
        a, b = predict_models(self.bundle, requests)
        result = sample.copy()
        result['rating'] = np.round((a + b) / 2)
        assert np.isfinite(result.rating).all() and (result.rating >= 0).all()
        assert result.prediction_id.equals(sample.prediction_id)
        return result

def replay_example(bundle, data_dir):
    p = Path(data_dir) / 'example_test_files'
    tables = [pd.read_csv(p / f) for f in
              ['test.csv', 'test_peptides.csv', 'test_proteins.csv', 'sample_submission.csv']]
    predictor = Predictor(bundle)
    results = []
    for key in sorted(tables[0].group_key.unique()):
        batch = [table.loc[table.group_key == key].copy() for table in tables]
        results.append(predictor.predict_batch(*batch))
    result = pd.concat(results, ignore_index=True)
    assert len(result) == len(tables[-1]) == 64
    print('Local chronological replay passed: 2 batches, 64 predictions.', flush=True)
    return result

def run_official_api(bundle):
    # Direct native API, as requested by the project's API_CALL_RULES.md.
    import amp_pd_peptide
    env = amp_pd_peptide.make_env()
    predictor = Predictor(bundle)
    count = 0
    for test, peptides, proteins, sample in env.iter_test():
        result = predictor.predict_batch(test, peptides, proteins, sample)
        env.predict(result)
        count += len(result)
    print('Official native API completed: {} predictions.'.format(count), flush=True)


## 完整性检查

In [ ]:
# Mathematical and temporal integrity checks, before real training.
assert smape1p([0, 1], [0, 1]) == 0
assert abs(smape1p([0], [1]) - 200 / 3) < 1e-10
for cls in (0, 1, 43, 86):
    probabilities = np.eye(87)[[cls]]
    assert optimal_integer(probabilities)[0] == cls
rng = np.random.RandomState(0)
probs = rng.dirichlet(np.ones(87), size=7)
brute = np.array([[sum(p[y] * 200 * abs(y-a)/(2+y+a) for y in range(87))
                   for a in range(87)] for p in probs]).argmin(axis=1)
assert np.array_equal(optimal_integer(probs), brute)
request = pd.DataFrame({'patient_id': [1,1,1,1], 'visit_month': [0,6,12,24], 'horizon': 6})
history = pd.DataFrame({'patient_id': [1,1,1,1], 'visit_month': [0,6,18,48]})
features = causal_features(request, history)
for i, row in request.iterrows():
    censored = history.loc[history.visit_month <= row.visit_month]
    expected = causal_features(request.loc[[i]], censored)
    pd.testing.assert_frame_equal(features.loc[[i]], expected)
assert features.seen_6.tolist() == [0,1,1,1]
assert features.seen_18.tolist() == [0,0,0,1]
assert features.previous_nonannual.tolist() == [0,1,1,2]
print('Metric, Bayes decision and future-history invariance checks passed.')


## 数据与运行配置

In [ ]:
# Kaggle uses the previously verified original native-API environment.
KAGGLE = Path('/kaggle').exists()
if KAGGLE:
    DATA_DIR = Path('/kaggle/input/amp-pd')
    RUN_DIR = Path('/kaggle/working')
else:
    # Notebook can be opened from either the project root or outputs/.
    ROOT = Path.cwd().parent if Path.cwd().name == 'outputs' else Path.cwd()
    DATA_DIR = ROOT / 'work/data'
    RUN_DIR = ROOT / 'outputs/origin_months_experiment/run'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_CV = not KAGGLE
print('data:', DATA_DIR, 'run CV:', RUN_CV)
print('versions:', {'numpy': np.__version__, 'pandas': pd.__version__,
                    'torch': torch.__version__, 'lightgbm': lgb.__version__})
visits = load_data(DATA_DIR)
samples = build_samples(visits)
print('visits:', len(visits), 'patients:', visits.patient_id.nunique(),
      'training requests:', len(samples), 'scored requests:', int(samples.score_eligible.sum()))
assert not set(TARGETS + ['patient_id', 'label']).intersection(FEATURES)

assert not ((visits.supplemental == 1) & (visits.visit_month == 5)).any()
assert set(samples.visit_month).issubset({0,6,12,18,24,36,48,60,72,84})
assert len(samples) == 36944
print('Requested preprocessing checks passed: 36,944 labeled requests.')


## 患者分组验证

In [ ]:
if RUN_CV:
    report, oof = cross_validate(samples, CONFIG)
    (RUN_DIR / 'validation.json').write_text(json.dumps(report, indent=2))
    oof.to_csv(RUN_DIR / 'oof.csv', index=False)
    print(json.dumps({k: report[k] for k in ['pooled_oof', 'mean_fold']}, indent=2))
else:
    print('Kaggle submission trains on all data; local grouped validation is saved separately.')


## 全量训练及本地时序样例

In [ ]:
bundle = fit_models(samples, CONFIG)
bundle[0].save_model(str(RUN_DIR / 'lightgbm.txt'))
example = replay_example(bundle, DATA_DIR)
# This is a local demonstration artifact, never used as the competition submission.
example.to_csv(RUN_DIR / 'example_predictions.csv', index=False)
print(example.head(8).to_string(index=False))


## 官方原生时序 API

In [ ]:
if KAGGLE:
    run_official_api(bundle)
else:
    print('Native Linux competition extension runs on Kaggle. Local replay checked separately.')
